# Antes del descenso de gradiente: así se entrena una regresión lineal en la práctica

Imagina que quieres calcular una suma larga. Sabes sumar a mano, dígito por
dígito — pero en la práctica usas una calculadora, porque ya alguien resolvió
ese problema de forma rápida y confiable. Con una regresión lineal pasa algo
parecido: puedes programar tú mismo el algoritmo que busca la mejor recta (lo
harás en [`01_funcion_de_costo.ipynb`](01_funcion_de_costo.ipynb), a propósito,
para entender el mecanismo), pero en un proyecto real casi siempre le pides a
una librería que lo resuelva por ti.

La pregunta de este notebook es concreta: **¿qué escribes en Python para
entrenar y usar de verdad una regresión lineal, y qué significa cada cosa que
te devuelve?** Los notebooks siguientes de esta carpeta abren el cofre y
programan el mecanismo interno a mano; aquí solo aprendemos a usar la
herramienta ya construida — la calculadora, no la suma dígito por dígito.

In [1]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

datos = pl.DataFrame({
    "horas_estudio": [0, 1, 2, 3, 4, 5, 6],
    "calificacion": [5, 12, 22, 26, 38, 43, 48],
})
datos

horas_estudio,calificacion
i64,i64
0,5
1,12
2,22
3,26
4,38
5,43
6,48


## 1. Qué le vas a pedir a la librería

Mira los puntos de este dataset sin ninguna recta encima todavía. Sabes que
existe alguna recta $\hat y = wx + b$ que se ajusta razonablemente bien, pero
no sabes sus valores exactos de $w$ y $b$. Eso es exactamente lo que le vas a
pedir a scikit-learn: que **busque** esos dos números por ti.

In [2]:
x = datos["horas_estudio"].to_numpy()
y = datos["calificacion"].to_numpy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="datos reales", marker={"size": 11, "color": "black"}))
fig.update_layout(
    title="Solo los datos: ¿qué recta los describe mejor?",
    xaxis_title="Horas de estudio", yaxis_title="Calificación",
)
fig.show()

## 2. El patrón que comparten (casi) todos los modelos de scikit-learn

Piensa en entrenar a alguien nuevo en un trabajo: primero le muestras casos ya
resueltos ("con estos síntomas, este fue el diagnóstico correcto"), y una vez
que aprendió el patrón, le presentas un caso nuevo y le pides una decisión. En
scikit-learn ese mismo proceso se llama, literalmente, así:

1. **Construir el modelo** — eliges qué tipo de "aprendiz" quieres (aquí,
   `LinearRegression()`), todavía sin datos.
2. **`.fit(X, y)`** — "entrena": le muestras las entradas `X` y las respuestas
   correctas `y`, y el modelo ajusta sus parámetros internos para minimizar el
   costo (el mismo MSE que ya viste, o verás, en
   [`01_funcion_de_costo.ipynb`](01_funcion_de_costo.ipynb)).
3. **`.predict(X_nuevo)`** — "pregunta": le das entradas nuevas, nunca vistas
   durante el entrenamiento, y te devuelve una predicción para cada una.

Un detalle práctico que suele confundir al principio: `X` debe ser una
**matriz** (una tabla de filas × columnas), no una simple lista de números,
incluso si solo tienes una columna de entrada. Scikit-learn siempre espera
"una fila por observación, una columna por variable de entrada", porque está
pensado para funcionar igual con 1 variable o con 100. Por eso convertimos
`horas_estudio` con `.reshape(-1, 1)`: significa "conviértete en una matriz de
una sola columna, calculando tú mismo cuántas filas hacen falta".

In [3]:
X = x.reshape(-1, 1)

modelo = LinearRegression()
modelo.fit(X, y)

print(f"w (modelo.coef_[0]):    {modelo.coef_[0]:.2f}")
print(f"b (modelo.intercept_):  {modelo.intercept_:.2f}")

w (modelo.coef_[0]):    7.39
b (modelo.intercept_):  5.54


`modelo.coef_` y `modelo.intercept_` son exactamente `w` y `b`: los mismos dos
números que en [`02_parametros_w_y_b.ipynb`](02_parametros_w_y_b.ipynb)
interpretamos como "pendiente" y "punto de partida". `coef_` es un array (una
lista) porque un modelo con más de una variable de entrada tendría un peso por
cada una; con una sola variable, tiene un único elemento.

Fíjate también en algo que no viste: no hubo que elegir una tasa de
aprendizaje ni un número de épocas, como sí ocurre con el descenso de
gradiente de [`01_funcion_de_costo.ipynb`](01_funcion_de_costo.ipynb). Para
una regresión lineal simple, scikit-learn no "camina" iterativamente hacia el
mínimo: lo calcula de forma directa con álgebra lineal (**mínimos
cuadrados**), porque para este problema existe una fórmula exacta.

### ¿Qué son "mínimos cuadrados"?

En `01_funcion_de_costo.ipynb` viste que el costo, al variar $w$, dibuja una
curva con forma de valle — y que el mínimo de esa curva es el único punto
donde la tangente queda perfectamente horizontal (pendiente/derivada igual a
cero). El descenso de gradiente encuentra ese punto **a tientas**: da un paso
pequeño cuesta abajo, mide la pendiente de nuevo, da otro paso, y así hasta
acercarse lo suficiente.

Mínimos cuadrados es la otra forma de llegar al mismo lugar: en vez de
caminar paso a paso, se plantea directamente la ecuación "¿para qué $w$ y qué
$b$ la pendiente es exactamente cero?" y se **despeja** — como cuando, en vez
de probar valores de $x$ hasta acertar en una ecuación, la resuelves de una
sola vez con álgebra. Es la diferencia entre sintonizar una radio girando el
dial poco a poco hasta oír la señal, y calcular de antemano el número exacto
al que hay que ponerlo.

Para una sola variable de entrada, esa ecuación tiene una solución conocida:

$$w = \frac{\sum_{i=1}^n (x_i - \bar x)(y_i - \bar y)}{\sum_{i=1}^n (x_i - \bar x)^2}, \qquad b = \bar y - w\bar x$$

Leyendo cada pieza:

- $\bar x$ y $\bar y$ (se leen "equis barra" y "y barra") son, simplemente,
  el **promedio** de cada columna.
- $(x_i - \bar x)$ es "cuánto se aleja ese dato de `x` de su propio
  promedio" — una **desviación**. Lo mismo para $(y_i - \bar y)$.
- El numerador suma el producto de ambas desviaciones: es grande y positivo
  cuando `x` e `y` se alejan de su promedio **juntos y en la misma
  dirección** (a más horas de estudio, más calificación); es la forma más
  simple de medir "qué tanto se mueven acompañados" dos variables.
- El denominador solo mide qué tan dispersos están los valores de `x` por sí
  solos, sin relación con `y`.
- $w$ es, entonces, "cuánto se mueven juntos `x` e `y`" dividido entre
  "cuánto se mueve `x` por su cuenta" — la pendiente que mejor sigue esa
  tendencia conjunta.
- $b$ se ajusta al final para que la recta pase exactamente por el punto
  $(\bar x, \bar y)$: el "centro" de la nube de datos.

Esta fórmula no salió de la nada: es el resultado de tomar las derivadas del
costo que ya viste en `01_funcion_de_costo.ipynb`
($\partial J/\partial w$ y $\partial J/\partial b$), igualar ambas a cero, y
despejar $w$ y $b$ del sistema de dos ecuaciones que queda. "Mínimos
cuadrados" es literalmente eso: la solución algebraica exacta de "¿dónde el
error cuadrático medio deja de poder bajar más?".

In [4]:
x_prom = x.mean()
y_prom = y.mean()

w_manual = np.sum((x - x_prom) * (y - y_prom)) / np.sum((x - x_prom) ** 2)
b_manual = y_prom - w_manual * x_prom

print(f"w calculado a mano (mínimos cuadrados): {w_manual:.6f}")
print(f"w que devolvió LinearRegression:        {modelo.coef_[0]:.6f}")
print(f"b calculado a mano (mínimos cuadrados): {b_manual:.6f}")
print(f"b que devolvió LinearRegression:        {modelo.intercept_:.6f}")

w calculado a mano (mínimos cuadrados): 7.392857
w que devolvió LinearRegression:        7.392857
b calculado a mano (mínimos cuadrados): 5.535714
b que devolvió LinearRegression:        5.535714


Coinciden exactamente (salvo el redondeo normal de punto flotante) — y no es
casualidad: es la misma fórmula. `LinearRegression` no usa por dentro
exactamente estas sumas (usa una variante numéricamente más estable basada en
SVD, útil cuando hay muchas variables de entrada), pero matemáticamente
resuelve la misma pregunta y llega al mismo resultado. El descenso de
gradiente sigue siendo útil (y necesario) para modelos donde esa fórmula
exacta no existe o es demasiado costosa de calcular — lo verás en varios
modelos de las próximas carpetas. La pregunta de "cuándo existe esa fórmula
exacta y cuándo no" se responde a fondo en
[`06_minimos_cuadrados_vs_descenso_de_gradiente.ipynb`](06_minimos_cuadrados_vs_descenso_de_gradiente.ipynb).

In [5]:
y_predicho = modelo.predict(X)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="datos reales", marker={"size": 11, "color": "black"}))
fig.add_trace(go.Scatter(
    x=x, y=y_predicho, mode="lines", name="recta encontrada por LinearRegression",
    line={"color": "#d62728", "width": 3},
))
fig.update_layout(
    title=f"scikit-learn encontró w={modelo.coef_[0]:.2f}, b={modelo.intercept_:.2f}",
    xaxis_title="Horas de estudio", yaxis_title="Calificación",
)
fig.show()

## 3. Usar el modelo ya entrenado

Con el modelo entrenado, `.predict()` funciona sobre cualquier entrada nueva,
aunque nunca haya aparecido en los datos de entrenamiento — por ejemplo, 3.5
horas de estudio, un valor que no está en la tabla original.

In [6]:
horas_nuevas = np.array([[3.5]])
calificacion_estimada = modelo.predict(horas_nuevas)
print(f"Calificación estimada para 3.5 horas de estudio: {calificacion_estimada[0]:.2f}")

r2 = modelo.score(X, y)
print(f"R² sobre los datos de entrenamiento: {r2:.3f}  (1.0 sería un ajuste perfecto)")

Calificación estimada para 3.5 horas de estudio: 31.41
R² sobre los datos de entrenamiento: 0.988  (1.0 sería un ajuste perfecto)


`.score()` en un modelo de regresión de scikit-learn devuelve **R²**, un
número (típicamente entre 0 y 1) que resume qué tan bien explica el modelo la
variación de `y` — 1.0 sería un ajuste perfecto. Aquí lo calculamos sobre los
mismos datos con los que entrenamos, solo como primera referencia rápida; R²
se explica a fondo, y se evalúa correctamente sobre datos de prueba separados,
en su propio notebook más adelante (fila "Métricas de regresión" en
[GLOSARIO.md](../../../../GLOSARIO.md)).

## 4. El mismo patrón, otros modelos

Esto es lo importante: **`LinearRegression`, `Ridge`, `HuberRegressor` y casi
cualquier otro modelo de scikit-learn se usan exactamente igual** —
`modelo.fit(X, y)` seguido de `modelo.predict(X_nuevo)` — aunque por dentro
cada uno minimice una función de costo distinta (MSE, MSE + penalización,
Huber Loss...). Cambiar de modelo casi nunca significa cambiar cómo lo usas,
sino solo qué tan robusto es o qué supuestos hace sobre tus datos.

In [7]:
from sklearn.linear_model import HuberRegressor, Ridge

modelo_ridge = Ridge(alpha=1.0).fit(X, y)
modelo_huber = HuberRegressor().fit(X, y.astype(float))

comparacion = pl.DataFrame({
    "modelo": ["LinearRegression (MSE)", "Ridge (MSE + penalización)", "HuberRegressor (Huber Loss)"],
    "w": [modelo.coef_[0], modelo_ridge.coef_[0], modelo_huber.coef_[0]],
    "b": [modelo.intercept_, modelo_ridge.intercept_, modelo_huber.intercept_],
})
comparacion

modelo,w,b
str,f64,f64
"""LinearRegression (MSE)""",7.392857,5.535714
"""Ridge (MSE + penalización)""",7.137931,6.300493
"""HuberRegressor (Huber Loss)""",7.347209,5.49021


Con datos tan ordenados y sin valores atípicos, las tres rectas quedan casi
idénticas — la elección de función de costo importa poco cuando no hay nada
difícil que resolver. La diferencia se vuelve visible cuando los datos tienen
outliers (ya viste ese caso con Huber en
[`01_funcion_de_costo.ipynb`](01_funcion_de_costo.ipynb) y lo verás a fondo en
[`03_huber_loss_en_profundidad.ipynb`](03_huber_loss_en_profundidad.ipynb)) o
muchas variables correlacionadas (el motivo de Ridge y Lasso, en una futura
lección de regularización).

## 5. Ideas clave

- Casi todos los modelos de scikit-learn siguen el mismo patrón: construir →
  `.fit(X, y)` → `.predict(X_nuevo)`.
- `X` siempre es una matriz (filas = observaciones, columnas = variables),
  incluso con una sola variable de entrada — de ahí el `.reshape(-1, 1)`.
- Los atributos que terminan en `_` (como `coef_` e `intercept_`) son los
  parámetros que el modelo *aprendió* al entrenar; no existen antes de llamar
  a `.fit()`.
- Para una regresión lineal simple, scikit-learn resuelve el ajuste con
  álgebra exacta, sin necesidad de descenso de gradiente — pero el mismo
  patrón `.fit`/`.predict` funciona igual para modelos donde sí hace falta.
- Cambiar de modelo (`LinearRegression` → `Ridge` → `HuberRegressor`) casi
  nunca cambia el código que lo usa, solo qué función de costo minimiza por
  dentro.

**Ejercicio:** añade un valor atípico a `datos` (por ejemplo, una fila con
`horas_estudio=6` pero `calificacion=150`, un error de captura) y vuelve a
ejecutar las secciones 2 y 4. Antes de mirar la tabla de comparación, predice:
¿el `w` de cuál modelo va a cambiar más — `LinearRegression` o
`HuberRegressor`? ¿Por qué?